# Published classification recipe: ZuCo SR random-split reconstruction

**CPU → Run all**, using **Zyion** for both Drive prompts. Finish any active notebook before changing its runtime type.

**This is a full, expensive model search—not the quick averaging sweep.** Default:40-,20-,10-,then1-trial conditions;100random splits each;10-fold inner validation; eight NCA component counts; full published C/gamma grids for logistic regression and RBF-SVM. Approximately **3,520,800 classifier fits** and up to **32,800 NCA fits**. NCA transforms are reused across C/gamma settings, avoiding unnecessary duplicate NCA fitting. Expect a long CPU workload, potentially multiple Colab sessions. There is no automatic smaller-grid or fixed-model fallback. scikit-learn NCA does not use a T4 GPU.

**Data:** saved **SST-labelled ZuCo SR** reconstruction, **not agent labels**, cleaned EEG,105channels ×63samples in0–500ms (actual grid0,8,...496ms),6615features. No new EEG extraction, source downloads or cleaning. The saved retained trials/preprocessing differ from the paper, so this cannot be claimed as an exact numerical reproduction.

**No identity holdouts:** average trials first, then repeat random95/5example splits100times. Reader, word and sentence overlap is permitted; each reader is represented on both sides when feasible. Test words are not required to be novel or familiar. Every original event belongs to exactly one average, so the same event never enters train and test. **Different fixation events can still contain overlapping EEG intervals; this protocol does not purge them.** Existing full-recording cleaning precedes splitting.

**Explicit reconstruction choices:** the paper does not provide the exact averaging memberships, seeds, split-rounding rule, solver settings or CI formula. Here, sort by word length with random tie order within reader/class, use distinct sentence texts within an average, and retain undersized remainders. For95/5splits, allocate integer test counts proportionally to reader/class margins. A~17-example test set cannot contain all24reader×class cells; some joint cells are absent. This keeps95/5rather than silently changing the test fraction. These choices are documented, not presented as recovered author code.

**Storage:** up to **500 MB total new Drive output**, including reusable6615-feature reader caches, trial-average caches and compact grid checkpoints. Dependencies and temporary EEG copies stay on Colab disk. Existing archives/results are unchanged. Preparation resumes per reader. Training resumes per completed inner-fold/NCA-component block and outer split; an interrupted block may need repeating. No fitted model is repeatedly overwritten on Drive.

**Interpretation:** primary metric is ordinary accuracy, as in the paper; balanced accuracy is also shown. Test labels construct same-class averages. These are offline group-level predictions, not individual-word accuracy. Repeated random splits are correlated, so the descriptive split-mean t interval is not independent-reader uncertainty or a verified match to the paper's unspecified CI calculation. Convergence warnings are counted for review.

Source: [Pfeiffer et al.(2020)](https://doi.org/10.1016/j.neuroimage.2020.116934), methods2.4.5and Table1.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


## 1. Connect and install the pinned cloud environment

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_paper_classification'
REPO=Path('/content/word-sentiment-eeg-paper-classification')
CODE_REVISION='0fef52f3f9c28cc184b6da6b1591daca09679c8d'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
ENV=Path('/content/word-eeg-paper-classification-python')
PYTHON=ENV/'bin/python'
if not PYTHON.exists():
    subprocess.run([sys.executable,'-m','venv',str(ENV)],check=True)
subprocess.run([str(PYTHON),'-m','pip','install','-q','-r',str(REPO/'requirements-paper-classification.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,500_000_000-used):
    raise RuntimeError('Reserve up to 500 MB for the feature cache and results. No EEG loading has started.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 500 MB.')
print('New output folder:',OUT)


## 2. Prepare reusable features and averages

Reads only saved accepted reader archives, one at a time. Keeps the SST labels and original accepted trials. This initial loading takes time; completed reader caches survive a runtime reset. Size1uses those caches directly to avoid a second full feature copy.

In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/zuco_paper_classification.py'),
         '--thesis-root',str(THESIS),'--phase','prepare',
         '--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='Paper classification preparation: saved EEG to reusable features')


## 3. Inspect actual sample counts and documented split allocation

In [ ]:
rows=[]
for size in [1,10,20,40]:
    saved=json.loads((OUT/f'averages/size{size:02d}/manifest.json').read_text())
    rows.append(dict(size=size,examples=saved['examples'],original_events=saved['original_events'],
                     remainders=saved['remainders'],minimum_group=saved['actual_min'],maximum_group=saved['actual_max']))
display(pd.DataFrame(rows))
allocation=json.loads((OUT/'averages/size40/split_allocation_example.json').read_text())
print('40-trial condition: test examples',allocation['test_count'],'| Actual test fraction:',round(allocation['test_fraction_actual'],4))
print('Allocation:',allocation['method'])
display(pd.DataFrame(allocation['allocation']))
print('Our counts need not equal the paper’s12,789 /1,300 /650 /328because the trial set and unspecified averaging details differ.')


## 4. Full published-grid classification

**This is the long step.** It starts with the40-trial condition. Each split has10inner folds; scaling and supervised NCA fit the inner training data only. Select NCA dimension/C/gamma by inner accuracy, then refit using the outer training set and evaluate the untouched5%test examples. C and gamma each span ten logarithmic values from1e−7to1e2; NCA dimensions10,20,40,80,160,320,640,1280. No class-weight modification. Completed grid blocks are reused. Progress reports each component/fold and completed block. You can interrupt and resume; a full run is not guaranteed to fit one free Colab session.

In [ ]:
command=[str(PYTHON),'-u',str(REPO/'scripts/zuco_paper_classification.py'),
         '--thesis-root',str(THESIS),'--phase','decode',
         '--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='Full paper classification grid:100random95/5splits, no identity holdouts')


## 5. Results and partial-run status

This cell can also show completed splits after an interruption. Do not treat partial results as the completed100-split analysis. The paper reference lines are accuracy benchmarks, not evidence that our different retained trial set exactly reproduces the study.

In [ ]:
import glob
rows=[]
for filename in sorted(OUT.glob('decoding/size*/repeat*/result.json')):
    result=json.loads(filename.read_text())
    size=int(filename.parents[1].name.removeprefix('size'))
    rows.extend(dict(size=size,**v) for v in result['scores'])
if rows:
    frame=pd.DataFrame(rows)
    display(frame.groupby(['size','model']).agg(completed_splits=('accuracy','size'),
        mean_accuracy=('accuracy','mean'),mean_balanced_accuracy=('balanced_accuracy','mean')))
else:
    print('No completed outer split yet. Feature and grid checkpoints may already be saved.')
if (OUT/'classification_comparison.png').exists():
    display(Image(filename=str(OUT/'classification_comparison.png')))
    display(pd.read_csv(OUT/'summary.csv'))
print('Full run complete:',(OUT/'complete.json').exists())
print('Scientific environment:',PYTHON)
print('Private output folder:',OUT)
